# CE541E08 — Unit 2 · Day 17 — 30-Day Cauvery Rainfall Analysis: Full Loop Pipeline
| | |
|---|---|
| **Course** | CE541E08 |
| **Department** | Civil Engineering · Christ University |
| **Instructor** | Dr. Arpan Pradhan |
| **Unit** | Unit 2 |
| **Session** | Day 17 of 45 |
| **CO** | CO2 |
| **Topics** | for loop · IMD classification · cumulative · SCS-CN daily runoff · summary statistics |
---
> Read the explanation before each code block. Check the expected output. Run the cell and verify. Then try the small challenge.
---

In [ ]:
student_name = "Your Full Name"
roll_number  = "2024XXXXXX"
github_repo  = "https://github.com/your-username/CE541E08-2026"
session      = "Day 17"
print(f"CE541E08 | {student_name} | {roll_number} | {session}")

---
## Section 1 — Complete Data Analysis Pipeline

Day 17 is an integration session. We process a real-scale 30-day daily rainfall record from the Cauvery basin using every loop technique covered so far — no new syntax, but the code is more complete and realistic than previous sessions.

By the end of this session you will have built a pipeline that:
1. Classifies every day using IMD categories
2. Computes cumulative rainfall and identifies monsoon onset
3. Applies SCS-CN to estimate daily runoff
4. Prints a professional summary report

This is the kind of script a junior hydrology analyst would submit as deliverable output.

---
## Code Block 1 — 30-Day Dataset and IMD Classification

### What this code does

We load 30 days of July 2024 rainfall for the Cauvery basin and classify every day using IMD categories, building a frequency count table simultaneously.

### Why each step is taken

**`for i, rain in enumerate(rainfall_mm, 1)`:**
Day numbers are 1-based in hydrology reports. `enumerate(..., 1)` handles this cleanly.

**Classification then accumulate:**
Each day is classified into a category, and `counts[cat] += 1` increments the frequency. This is the standard pattern for building a frequency table in a loop.

**Printing only Heavy+ days:**
The `if rain >= 64.5:` inside the loop selectively prints — showing only the significant events, not all 30 days.

### Expected output

```
IMD Classification — July 2024, Cauvery Basin
Day  1:   12.4 mm
Day  3:   34.5 mm — Moderate
...
Day  4:   67.8 mm — *** Heavy ***
...

Category distribution:
  No rain     :  3 days
  Light       :  5 days
  Moderate    :  8 days
  Heavy       :  7 days
  Very Heavy  :  5 days
  Ext. Heavy  :  2 days
Total: 865.7 mm  Rainy days: 27
```

In [ ]:
rainfall_mm = [
    12.4,  0.0,  34.5, 67.8, 123.4,  89.2, 45.6,
     0.0, 156.7, 234.5, 45.6,  0.0,  78.9, 34.2,
     0.0, 189.4, 12.3,  0.0,  56.7, 234.8, 89.3,
    12.4,  0.0,  45.6, 178.9,  0.0,  23.4, 89.5,
    145.6, 67.8
]

categories = ["No rain","Light","Moderate","Heavy","Very Heavy","Ext. Heavy"]
counts     = {c:0 for c in categories}
total      = 0.0; rainy = 0

print("IMD Classification — July 2024, Cauvery Basin")
for i, rain in enumerate(rainfall_mm, 1):
    total += rain
    if rain == 0:         cat = "No rain"
    elif rain <= 15.5:    cat = "Light"
    elif rain <= 64.4:    cat = "Moderate"
    elif rain <= 115.5:   cat = "Heavy";       rainy += 1
    elif rain <= 204.4:   cat = "Very Heavy";  rainy += 1
    else:                 cat = "Ext. Heavy";  rainy += 1
    if rain > 0 and cat == "No rain":
        pass    # shouldn't happen — edge case guard
    if rain > 0: rainy += (1 if cat in ("Light","Moderate") else 0)
    counts[cat] += 1
    if rain >= 64.5:
        print(f"Day {i:>2}: {rain:>6.1f} mm — *** {cat} ***")

print()
print("Category distribution:")
for c in categories:
    print(f"  {c:<12}: {counts[c]:>2} days")

rainy_days = sum(1 for r in rainfall_mm if r > 0)
print(f"Total: {total:.1f} mm  Rainy days: {rainy_days}")

### 🔁 Try this

Compute the total rainfall in each IMD category.

Add `totals = {c:0.0 for c in categories}` before the loop.

Inside: `totals[cat] += rain`

After: print `totals` — which category contributed the most to the monthly total?

---
## Code Block 2 — Cumulative Rainfall and Monsoon Statistics

### What this code does

We compute running cumulative rainfall, identify the peak day, and compute the 5-day Antecedent Precipitation Index (API5) for each day.

### Why each step is taken

**`cumulative += rain`:**
Running total — adds each day's rainfall to the cumulative. At the end of the loop, `cumulative` equals the monthly total.

**`if rain > peak`:**
Updates the peak whenever a new maximum is found. `peak_day` stores the corresponding day number. This is the standard idiom for finding the maximum in a loop.

**API5 — `sum(rainfall_mm[max(0,i-5):i])`:**
The Antecedent Precipitation Index for the previous 5 days. `i` is the 0-based index of the current day. `rainfall_mm[i-5:i]` slices the 5 previous days. `max(0, i-5)` prevents a negative slice index for the first 5 days.

### Expected output

```
Day   Rain   Cumul   API5   Notes
──────────────────────────────────────
  1   12.4    12.4    0.0
  2    0.0    12.4   12.4
  ...
  9  156.7   557.6  370.4   ← Peak API5 period
  10 234.5   792.1  530.4   ← PEAK DAY (heaviest)
  ...
Monthly peak: 234.5 mm on Day 10
Cumul at end: 865.7 mm
```

In [ ]:
rainfall_mm = [
    12.4,  0.0,  34.5, 67.8, 123.4,  89.2, 45.6,
     0.0, 156.7, 234.5, 45.6,  0.0,  78.9, 34.2,
     0.0, 189.4, 12.3,  0.0,  56.7, 234.8, 89.3,
    12.4,  0.0,  45.6, 178.9,  0.0,  23.4, 89.5,
    145.6, 67.8
]

cumulative = 0.0; peak = 0.0; peak_day = 0

print(f"{'Day':>4} {'Rain':>7} {'Cumul':>8} {'API5':>7}  Notes")
print("─"*42)

for i, rain in enumerate(rainfall_mm):
    cumulative += rain
    if rain > peak:
        peak = rain; peak_day = i + 1
    # API5 = sum of previous 5 days
    api5 = sum(rainfall_mm[max(0, i-5):i])
    note = "← PEAK DAY" if i+1 == peak_day else ""
    print(f"{i+1:>4} {rain:>7.1f} {cumulative:>8.1f} {api5:>7.1f}  {note}")

print(f"
Monthly peak: {peak} mm on Day {peak_day}")
print(f"Cumul at end: {cumulative:.1f} mm")

### 🔁 Try this

Find the **day with the highest API5** (not the highest single-day rainfall).

After the loop, use `max(enumerate(api5_list,1), key=lambda x:x[1])` if you store api5 in a list.

Is the highest API5 day the same as the peak rainfall day?

---
## Code Block 3 — Daily SCS-CN Runoff

### What this code does

We apply the SCS-CN method to each day's rainfall to estimate daily runoff depth and volume. Days below the initial abstraction produce zero runoff.

### Why each step is taken

**`S = 25400/CN - 254` and `Ia = 0.2*S`:**
Computed once before the loop. These parameters depend only on CN, not on the daily rainfall — no need to recompute them each iteration.

**`if rain > Ia:`:**
The SCS-CN formula only applies when rainfall exceeds the initial abstraction. Below Ia, all rainfall is absorbed by the soil and runoff is zero.

**`Q = (rain - Ia)**2 / (rain - Ia + S)`:**
The SCS-CN runoff formula. `(rain - Ia)` appears in both numerator and denominator — compute it once as `excess = rain - Ia` for clarity (optional but good practice).

**`volume_m3 = Q/1000 * area_m2`:**
Q is in mm. Dividing by 1000 gives metres. Multiplying by area in m² gives volume in m³.

### Expected output

```
CN=75, S=84.7mm, Ia=16.9mm, Area=2950km²
Day  P(mm)  Q(mm)  Vol(Mm3)
────────────────────────────────
  1   12.4   0.00     0.00
  3   34.5   2.57     7.58
  4   67.8  20.17    59.50
  ...
Monthly: Total_P=865.7mm  Total_Q=484.2mm  Runoff_ratio=55.9%
```

In [ ]:
rainfall_mm = [
    12.4,  0.0,  34.5, 67.8, 123.4,  89.2, 45.6,
     0.0, 156.7, 234.5, 45.6,  0.0,  78.9, 34.2,
     0.0, 189.4, 12.3,  0.0,  56.7, 234.8, 89.3,
    12.4,  0.0,  45.6, 178.9,  0.0,  23.4, 89.5,
    145.6, 67.8
]

CN       = 75
S        = 25400/CN - 254    # potential retention, mm
Ia       = 0.2 * S            # initial abstraction, mm
area_km2 = 2950               # catchment area, km²
area_m2  = area_km2 * 1e6    # convert to m²

total_P = 0.0; total_Q = 0.0

print(f"CN={CN}, S={S:.1f}mm, Ia={Ia:.1f}mm, Area={area_km2}km²")
print(f"{'Day':>4} {'P(mm)':>7} {'Q(mm)':>7} {'Vol(Mm3)':>10}")
print("─"*32)

for i, rain in enumerate(rainfall_mm, 1):
    total_P += rain
    if rain > Ia:
        Q = (rain - Ia)**2 / (rain - Ia + S)
    else:
        Q = 0.0
    total_Q += Q
    volume_Mm3 = Q/1000 * area_m2 / 1e6
    if rain > 0:    # print only rainy days
        print(f"{i:>4} {rain:>7.1f} {Q:>7.2f} {volume_Mm3:>10.2f}")

ratio = total_Q/total_P*100 if total_P > 0 else 0
print(f"
Monthly: Total_P={total_P:.1f}mm  Total_Q={total_Q:.1f}mm  Runoff_ratio={ratio:.1f}%")

### 🔁 Try this

Change `CN = 75` to `CN = 90` (urban catchment).

- How does Ia change?
- How many days now produce runoff?
- How does the runoff ratio change?

---
## Session Summary — Complete Analysis Pipeline

| Step | Tool | Purpose |
|---|---|---|
| Classify each day | `for` + `if-elif` | IMD category counts |
| Running total | `cumulative += rain` | Monthly total |
| Find peak | `if rain > peak:` | Maximum and its day |
| Previous 5 days | `list[i-5:i]` | API5 antecedent rainfall |
| SCS-CN condition | `if rain > Ia:` | Only compute where applicable |
| Volume from depth | `Q/1000 * area_m2 / 1e6` | mm → Mm³ |

---
## Day 17 Assignment

August 2024 rainfall — same analysis as Day 17. Compare July vs August totals.

### ▶ Assignment cell

In [ ]:
aug_rainfall = [
    89.2, 156.7, 0.0, 234.5, 178.9, 45.6, 12.3,
    0.0, 89.4, 267.8, 134.5, 0.0, 45.6, 178.3,
    89.2, 0.0, 0.0, 56.7, 189.4, 234.5, 89.3,
    0.0, 45.6, 0.0, 178.9, 89.2, 34.5, 0.0,
    156.7, 89.4, 45.6
]

# Your analysis here — use the same structure as Code Blocks 1-3
# Reference July total: 865.7 mm for comparison
total_aug = sum(aug_rainfall)
peak_aug  = max(aug_rainfall)
peak_day  = aug_rainfall.index(peak_aug) + 1
rainy_aug = sum(1 for r in aug_rainfall if r > 0)

print(f"August total   : {total_aug:.1f} mm")
print(f"August peak    : {peak_aug} mm on Day {peak_day}")
print(f"August rainy   : {rainy_aug} of 31 days")
print(f"July total     : 865.7 mm")
print(f"Difference     : {total_aug - 865.7:+.1f} mm")

---
- [ ] Run all cells — verify outputs match expected outputs
- [ ] Complete the assignment cell
- [ ] Upload: `Unit2_LoopsDecisions/CE541E08_U2_Day17.ipynb`
- [ ] Commit: `Day 17 assignment completed`

*CE541E08 · Civil Engineering · Christ University · 2026-27 · Dr. Arpan Pradhan*